## 6. Modelo Transformer preentrenado

Trabajo de María José Vire. Fase 1C: infraestructura y preanálisis del tokenizer.
El objetivo de esta fase es determinar la longitud máxima usando exclusivamente **train**.
Todavía no se ha realizado fine-tuning ni se presentan resultados de clasificación.

### 6.1 Definición del enfoque

Clasificación binaria de sentimiento en español con **BETO cased**, modelo base
`dccuchile/bert-base-spanish-wwm-cased`, y su tokenizer de Hugging Face.
La revisión del repositorio remoto queda fijada en la configuración.

- Única entrada: `comentario_limpio`.
- Etiqueta: `sentimiento_id`; Negativo = 0, Positivo = 1.
- `nombreAspecto`, `Valor`, `comentario` e `indice_original` no son predictores.
  `indice_original` se conserva únicamente para trazabilidad.
- Se reutilizan exactamente las particiones verificadas de Fase 0, comunes al baseline de Diego.
- Métrica principal futura: F1-macro. Train se utilizará para ajuste y validation para selección.
- Test permanece reservado para la fase final común; aquí solo se comprueba su integridad
  y su estructura. No se tokeniza, ni se examinan ejemplos para tomar decisiones.

**Entorno:** usar las dependencias existentes de Fase 0 y `requirements-transformer.txt`.
Transformers 4.57.1 publica compatibilidad con Python 3.12
([PyPI](https://pypi.org/project/transformers/4.57.1/)). La instalación se hace fuera de este
notebook; ninguna celda instala paquetes. PyTorch y la configuración CPU/GPU del entrenamiento
se decidirán posteriormente en Colab. Las semillas preparan reproducibilidad básica,
sin garantizar determinismo total.

In [1]:
from pathlib import Path
import sys

# Funciona desde la raíz o desde notebooks/, sin rutas personales.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/fase0.py").is_file()
             and (p / "configs/transformer.json").is_file()), None)
if root is None:
    raise RuntimeError("Abre el notebook dentro del repositorio con Fase 0 disponible.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
from IPython.display import display
from scripts.transformer import (
    load_config, set_seed, load_verified_partitions, load_tokenizer,
    tokenizer_metadata, analyze_train_lengths, record_max_length, save_analysis,
)

config_path = root / "configs/transformer.json"
config = load_config(config_path)
seed_status = set_seed(config["seed"])
print("Modelo:", config["model_id"])
print("Semillas:", seed_status)

Modelo: dccuchile/bert-base-spanish-wwm-cased
Semillas: {'seed': 42, 'python_random': True, 'numpy': True, 'torch': False}


### 6.2 Carga y verificación de las particiones

`load_verified_partitions()` utiliza exclusivamente `scripts.fase0.cargar_particiones()`.
Se comprueban hashes, tamaños, etiquetas, esquema y separación de las particiones mediante
la interfaz oficial. No se repiten la limpieza, deduplicación ni división del corpus.
Los siguientes recuentos son comprobaciones descriptivas de la interfaz, no métricas predictivas.

In [2]:
partitions = load_verified_partitions(root)
partition_summary = pd.DataFrame([
    {"partición": name, "observaciones": len(frame),
     "Negativo (0)": int(frame["sentimiento_id"].eq(0).sum()),
     "Positivo (1)": int(frame["sentimiento_id"].eq(1).sum())}
    for name, frame in partitions.items()
])
display(partition_summary)
for name, frame in partitions.items():
    print(name, "columnas:", list(frame.columns))
print("Tamaños esperados: train=16186, validation=3468, test=3469: verificados.")
print("Etiquetas {0, 1} e integridad Fase 0: verificadas.")
print("Entrada exclusiva:", config["text_column"], "| Etiqueta:", config["target_column"])
print("indice_original conservado únicamente para trazabilidad.")

,partición,observaciones,Negativo (0),Positivo (1)
0,train,16186,6870,9316
1,validation,3468,1472,1996
2,test,3469,1473,1996


train columnas: ['indice_original', 'comentario', 'Valor', 'nombreAspecto', 'comentario_limpio', 'sentimiento_id']
validation columnas: ['indice_original', 'comentario', 'Valor', 'nombreAspecto', 'comentario_limpio', 'sentimiento_id']
test columnas: ['indice_original', 'comentario', 'Valor', 'nombreAspecto', 'comentario_limpio', 'sentimiento_id']
Tamaños esperados: train=16186, validation=3468, test=3469: verificados.
Etiquetas {0, 1} e integridad Fase 0: verificadas.
Entrada exclusiva: comentario_limpio | Etiqueta: sentimiento_id
indice_original conservado únicamente para trazabilidad.


### 6.3 Selección y carga del tokenizer de BETO

Se usa `AutoTokenizer.from_pretrained` con el identificador y revisión de la configuración,
sin ejecutar código remoto. Se descargan exclusivamente archivos necesarios para tokenización;
**no se carga el clasificador ni pesos del modelo**. La clase y el vocabulario se registran
según el tokenizer realmente cargado, en lugar de inferirlos del nombre del modelo.

La caché se ubica en `results/transformer/tokenizer_cache/`, ignorada por Git.
La primera ejecución requiere acceso al Hub; las siguientes pueden usar la caché local.

In [3]:
tokenizer = load_tokenizer(config, root)
tokenizer_info = tokenizer_metadata(tokenizer, config)
display(pd.DataFrame([
    {"propiedad": key, "valor": str(value)}
    for key, value in tokenizer_info.items()
]))

/home/majovire/Escritorio/Visual Studio Code/Ciencia de Datos/Herramientas de IA/pln-sentimiento-estudiantil/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


,propiedad,valor
0,model_id,dccuchile/bert-base-spanish-wwm-cased
1,tokenizer_id,dccuchile/bert-base-spanish-wwm-cased
2,revision,c4d86612f51b4f46759c8390d1798c2febe71b93
3,class,BertTokenizerFast
4,is_fast,True
5,model_max_length,512
6,vocab_size,31002
7,vocab_with_added_tokens,31002
8,special_tokens,"{'unk_token': '[UNK]', 'sep_token': '[SEP]', '..."
9,special_token_ids,"{'[UNK]': 3, '[SEP]': 5, '[PAD]': 1, '[CLS]': ..."


### 6.4 Análisis de longitud tokenizada

La función vuelve a cargar las particiones por la interfaz oficial y extrae solamente **train**.
Tokeniza `comentario_limpio` sin truncamiento y sin padding, incluyendo los tokens especiales
que acompañarán cada secuencia individual. Los lotes de tokenización solo limitan memoria.
Los percentiles utilizan interpolación lineal de NumPy.

Regla fijada antes de observar las longitudes: elegir el menor candidato de
`[64, 128, 256, 512]` que cubra al menos el 99 % de train. Una secuencia de longitud igual
al límite está cubierta. Si ningún candidato cumple, elegir 512 e informar el porcentaje
que se truncaría. Validation y test no intervienen en esta decisión.

In [4]:
analysis = analyze_train_lengths(tokenizer, config, root)
print("Partición analizada:", analysis["split"], "| Observaciones:", analysis["n_observations"])
stat_names = {"min": "Mínimo", "mean": "Media", "median": "Mediana",
              "p90": "P90", "p95": "P95", "p99": "P99", "max": "Máximo"}
display(pd.DataFrame([
    {"estadístico": stat_names[key], "tokens": value}
    for key, value in analysis["statistics"].items()
]))
coverage_table = pd.DataFrame(analysis["coverage"]).rename(columns={
    "candidate": "límite candidato", "exceeds_count": "cantidad que excede",
    "exceeds_percent": "% que excede", "covered_percent": "% cubierto",
})
display(coverage_table)
print("max_length seleccionado:", analysis["max_length"])
print("Cobertura de al menos el 99 %:", analysis["coverage_target_met"])
print("Porcentaje que se truncaría:", analysis["selected_truncation_percent"])
if not analysis["coverage_target_met"]:
    print("Ningún candidato alcanza el 99 %; se utiliza el máximo permitido, 512.")

Token indices sequence length is longer than the specified maximum sequence length for this model (610 > 512). Running this sequence through the model will result in indexing errors


Partición analizada: train | Observaciones: 16186


,estadístico,tokens
0,Mínimo,3.000000
1,Media,20.284011
2,Mediana,14.000000
3,P90,38.000000
4,P95,52.000000
5,P99,103.150000
6,Máximo,610.000000


,límite candidato,cantidad que excede,% que excede,% cubierto
0,64,494,3.052020,96.947980
1,128,100,0.617818,99.382182
2,256,17,0.105029,99.894971
3,512,1,0.006178,99.993822


max_length seleccionado: 128
Cobertura de al menos el 99 %: True
Porcentaje que se truncaría: 0.6178178672927221


**Registro explícito de la decisión y del resumen descriptivo.** La siguiente celda actualiza
`configs/transformer.json` únicamente después del análisis real de train y guarda
`results/transformer/token_length_analysis.json`. El análisis por sí solo no modifica la
configuración. Se verifican el protocolo, la revisión del tokenizer y los hashes de train
antes de registrar la longitud. El resumen no incluye textos, predicciones ni métricas
 de clasificación.

Los resultados y la caché están ignorados por Git. Para reproducir desde la raíz:
`python -m scripts.transformer` (solo resumen); para registrar explícitamente la decisión:
`python -m scripts.transformer --record-max-length`. Si la caché está disponible, puede añadirse
`--local-files-only`. Este notebook también reproduce el análisis ejecutando sus celdas en orden.
Las particiones y metadatos locales de Fase 0 deben estar disponibles; no se regeneran aquí.

In [5]:
config = record_max_length(analysis, config_path, root)
analysis_path = save_analysis(analysis, root / "results/transformer/token_length_analysis.json")
print("Decisión registrada explícitamente:", config["max_length"])
print("Resumen descriptivo:", analysis_path.relative_to(root))

Decisión registrada explícitamente: 128
Resumen descriptivo: results/transformer/token_length_analysis.json


### 6.5 Preparación del protocolo experimental

Decisiones persistidas para el futuro fine-tuning:

- Seed: 42; máximo de 3 épocas.
- Learning rates a comparar: `1e-5`, `2e-5`, `3e-5`.
- Weight decay: 0.01; batch efectivo objetivo: 16.
- Evaluación al finalizar cada época, exclusivamente sobre validation.
- Selección: mayor F1-macro de validation; empate a cuatro decimales de F1-macro:
  mayor accuracy; si continúa, menor learning rate; si continúa, checkpoint de época más temprana.
- Test reservado para la fase final común.
- Longitud máxima derivada exclusivamente de train por la regla del 99 %.
- Padding dinámico por batch mediante `DataCollatorWithPadding` de Transformers.
  No se rellena todo el corpus hasta `max_length` en este preanálisis.

El pipeline de Fase 2A prepara dos combinaciones operativas: batch físico 16 /
acumulación 1, o físico 8 / acumulación 2. Se elegirá una posteriormente según la GPU
real de Colab, manteniéndola común entre experimentos. Dispositivo y precisión siguen
pendientes de esa inspección. Evaluación y guardado por época, retención de dos
checkpoints y recarga del mejor están implementados para su ejecución futura. Los checkpoints futuros se ubicarán en
`models/transformer/` y los resultados en `results/transformer/`; ambas rutas están ignoradas
por Git. El baseline de Diego permanece en su notebook y artefactos originales. La comparación
final se preparará en una fase posterior.

**Estado actual: el pipeline está implementado, pero no se ha cargado ni entrenado
ningún clasificador Transformer. T1/T2/T3 y el smoke real no se han ejecutado.**

### 6.6 Preparación de train y validation para fine-tuning

`load_training_frames()` carga la interfaz oficial de Fase 0 y devuelve únicamente train y
validation. Fase 0 puede verificar la integridad de test, pero test se descarta antes de
preparar el entrenamiento. No se cambian limpieza, particiones ni columnas persistidas.

`prepare_training_datasets(tokenizer, config)` convierte esas dos particiones a datasets
map compatibles con el DataLoader de PyTorch. Solo tokeniza `comentario_limpio`, con
`max_length=128`, `truncation=True`, tokens especiales y `padding=False`.
`DataCollatorWithPadding` realiza padding dinámico al formar cada batch.

Cada registro predictivo contiene únicamente `input_ids`, `attention_mask`, opcionalmente
`token_type_ids`, y `labels` procedente de `sentimiento_id`. `indice_original` se conserva
como metadato separado de trazabilidad, junto con el split y el hash del CSV; nunca entra
al modelo. `nombreAspecto`, `Valor` y `comentario` tampoco entran al modelo.

`guard_development_dataset()` verifica la procedencia, orden de índices, etiquetas, hashes y
revisión. El Trainer protegido controla entrenamiento, evaluación, dataloaders y `predict`:
solo train para ajuste y validation para evaluación/predicciones de desarrollo. Test, datasets
externos o registros renombrados como otro split se rechazan. No se utiliza el conjunto de
prueba para seleccionar checkpoints o hiperparámetros.

### 6.7 Configuración del futuro fine-tuning y checkpoints

`initialize_original_model()` utiliza `AutoModelForSequenceClassification.from_pretrained`
con BETO original y su revisión registrada. Establece `num_labels=2`,
`id2label={0: "Negativo", 1: "Positivo"}` y el `label2id` inverso. La cabeza de clasificación
se inicializará nueva; no se parte de un BETO afinado para sentimiento.

`training_argument_values()` permite revisar el plan sin cargar pesos.
`build_training_arguments()` y `build_trainer()` se invocarán posteriormente en Colab:
construir el Trainer ya inicializa el modelo, aunque no llama a `train()`.
Se exigen learning rate, batch físico y precisión explícitos; estas celdas no construyen el
Trainer ni ejecutan T1/T2/T3.

Se usan los nombres reales de Transformers **4.57.1**: `eval_strategy="epoch"` y
`processing_class=tokenizer`, comprobados contra el código instalado y la
[documentación oficial de Trainer](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer).
El nombre histórico `evaluation_strategy` se conserva solo como dato del protocolo JSON.

- Evaluación y guardado por época; máximo tres épocas para futuros experimentos oficiales.
- `load_best_model_at_end=True`, `metric_for_best_model="f1_macro"`, `greater_is_better=True`.
- `save_total_limit=2` y checkpoints en safetensors, con retención del mejor y del más reciente.
- Dentro de un experimento, empates en F1-macro a cuatro decimales se resuelven por accuracy;
  si persisten, se conserva la época más temprana. El learning rate es constante dentro del run.
- Cada experimento tiene un directorio nuevo `models/transformer/T1`, `T2` o `T3`.
  Se rechazan directorios existentes, reanudaciones y reutilización de un Trainer ya ejecutado.
- Cada `model_init` carga exclusivamente BETO original. El mejor checkpoint se recarga al
  finalizar su propio entrenamiento, sin usarlo para inicializar otro experimento.
- No hay early stopping ni búsqueda automática de hiperparámetros.
- Se preparan batch físico 16 / acumulación 1 y físico 8 / acumulación 2; ambos tienen
  batch efectivo 16 en un único proceso/dispositivo. GPU múltiple visible se rechaza para
  evitar multiplicar ese batch. La combinación operativa debe mantenerse común entre T1/T2/T3.
- Optimizador AdamW de PyTorch, scheduler lineal y cero pasos de warmup, registrados en el
  manifiesto; no se introduce una búsqueda adicional sobre estos parámetros.

`compute_metrics()` calcula F1-macro, accuracy, precision-macro y recall-macro con las clases
fijas `[0,1]`, `zero_division=0` y argmax de logits. También puede devolver precisión, recall,
F1 y soporte por clase. Las pruebas locales usan datos sintéticos, no logits reales del corpus.

El manifiesto de cada ejecución futura registra configuración, argumentos efectivos,
versiones, GPU/CUDA, checkpoint original, hashes y los índices de sus datasets. La comparación
entre T1/T2/T3 y el desempate por menor learning rate pertenecen a una fase posterior;
no se han seleccionado hiperparámetros.

### 6.8 Detección de dispositivo y preparación de Colab

La siguiente celda solo inspecciona el entorno: no descarga pesos ni crea un Trainer.
`detect_runtime()` informa versión de Python, PyTorch y Transformers, CUDA del runtime de
PyTorch y GPUs visibles con su memoria y capacidad de cómputo. Si PyTorch falta, lo informa
sin instalarlo y deja dispositivo/capacidades sin disponibilidad de entrenamiento.

La precisión se proporciona explícitamente al futuro pipeline: `fp32`, `fp16` o `bf16`.
CPU admite fp32 en este protocolo; CUDA permite fp16 según capacidad y bf16 solo con soporte
nativo comprobado mediante PyTorch. No se elige mixed precision automáticamente ni se
persiste ahora una opción definitiva. Las semillas cubren Python, NumPy y PyTorch/CUDA,
sin garantizar determinismo total.

**Dependencias futuras:** conservar Transformers 4.57.1, instalar Accelerate 1.10.1 y una
build de PyTorch compatible con el runtime. El checkpoint original de BETO usa `.bin`;
el cargador de Transformers exige **PyTorch >=2.6** para esa lectura. No se fija una build
CUDA local en `requirements-transformer.txt`.

En Colab se deberá comprobar primero PyTorch/CUDA/GPU y, si hace falta, instalar la build
indicada por el [selector oficial de PyTorch](https://pytorch.org/get-started/locally/).
Después instalar las dependencias de Fase 0 y `requirements-transformer.txt`; registrar las
versiones efectivas y disponer de las particiones y metadatos comunes, ignorados por Git.
No se requiere `datasets` ni `evaluate`: se usa un Dataset map y las métricas sklearn existentes.

In [ ]:
from scripts.transformer import detect_runtime
runtime_info = detect_runtime()
display(pd.DataFrame([
    {"propiedad": key, "valor": str(value)}
    for key, value in runtime_info.items()
]))
# No construir Trainer ni ejecutar train() en esta fase.

### 6.9 Smoke test técnico explícito — NO OFICIAL

`run_smoke_test()` queda implementado para ejecutarse posteriormente en el entorno adecuado,
con learning rate, batch físico y precisión indicados explícitamente. También se accede con
`python -m scripts.transformer --smoke-test`, acompañado de `--learning-rate`,
`--physical-batch-size`, `--precision` y opcionalmente `--device`.
No existe un comando CLI que lance T1/T2/T3.

El smoke toma un subconjunto determinista con seed 42: **32 observaciones de train**
(16 por clase) y **16 de validation** (8 por clase), conservando su trazabilidad.
Ejecuta exactamente **dos pasos de optimización**, como máximo una época pequeña,
y verifica forward, backward, evaluación, métricas, guardado por época y carga del mejor.
Comprueba también una actualización efectiva de los parámetros de la cabeza de clasificación.
Guarda ese modelo técnico, lo recarga localmente y comprueba sus logits sobre un pequeño
batch de validation sin persistir predicciones. Esta recarga sirve exclusivamente para
comprobar serialización; nunca inicializa un experimento oficial.

Los archivos se separan en `models/transformer/smoke/SMOKE_<id>/` y
`results/transformer/smoke/SMOKE_<id>/smoke_test_NO_OFICIAL.json`.
El informe tiene `official=false` y no puede utilizarse para comparar modelos ni seleccionar
learning rate, batch o precisión. La configuración experimental original no se modifica.
Test nunca se convierte, tokeniza, evalúa ni predice en este modo.

**El smoke real no se ha ejecutado localmente porque faltan PyTorch y Accelerate.
No hay resultados de entrenamiento ni métricas oficiales T1/T2/T3 en este notebook.**

### 6.10 Fase 3A — experimentos oficiales preparados, sin ejecutarlos

Fase 2B está cerrada: el smoke real de Colab pasó en Tesla T4, fp16, batch físico 16
/acumulación 1, sin uso predictivo de test. Véase
[el registro de Fase 2B](../docs/colab_smoke_transformer.md). Sus resultados siguen siendo NO OFICIALES.

**Esta sección sustituye las decisiones operativas históricas de 6.5–6.9**: oficialmente
se fijan CUDA/fp16/16/1, tres épocas y retención de tres checkpoints. T1/T2/T3 siguen sin ejecutarse.
Toda la lógica reside en `scripts/transformer.py`; este notebook no crea ni entrena un Trainer.

Posteriormente, desde la raíz en Colab, ejecutar de forma explícita y secuencial:

```bash
python -m scripts.transformer --experiment T1
python -m scripts.transformer --experiment T2
python -m scripts.transformer --experiment T3
```

La CLI fija T1=1e-5, T2=2e-5 y T3=3e-5; rechaza overrides de entrenamiento.
Cada invocación genera un UUID y parte de BETO original con la revisión del JSON,
sin resume ni sobrescritura. Train y validation proceden de Fase 0; test permanece reservado.

`load_best_model_at_end=False` y `metric_for_best_model=None` desactivan la selección HF oficial.
`select_validation_candidate` es la única selección de época y experimento:
`round(F1-macro,4)` descendente, accuracy descendente, LR ascendente, época ascendente.
El checkpoint elegido se carga por separado con `load_selected_checkpoint`, sin evaluar datos.

Leer `results/transformer/<T1|T2|T3>/<RUN_uuid>/validation_result.json` para metadata técnica,
épocas y `selected`. Tras los tres runs, `--summarize` recibe explícitamente sus tres archivos
para crear `results/transformer/validation_summary.json`, sin cargar modelos ni usar test.
El [protocolo de Fase 3A](../docs/fase3a_transformer.md) detalla comandos, esquema y requisitos
previos de Colab. Ningún comando de entrenamiento se ejecuta desde estas celdas.
